# CCI V2.1-P - Frozen V2 Package (Kaggle execution)

Steps 5 and 6 of the seven-step cycle, under **ADR-013**. Step 5 ratifies
the candidate selected by D1 and unchallenged by D2. Step 6 refits that
candidate and freezes it as a persisted package.

**Why this runs on Kaggle and not locally.** The word+char union over the
345,552-row `inner_fit` scope peaked at 10.2 GB RSS on the local machine
and aborted twice on low memory, which is why D1 moved here in the first
place. Just as important, D1's published numbers were produced in *this*
image, so this is the only environment where an exact reproduction check
is a fair test.

**The reproduction gate.** No fitted model was persisted during D1, so
freezing requires a refit, and a refit could silently differ from what was
measured. This run therefore refits along D1's exact code path and then
compares, with no numeric tolerance, the calibrated threshold, both full
confusion matrices, both override-decision pairs, and the hard-negative
pool counts. All checks pass and the bundle is written; any check fails
and the outcome is `REPRODUCTION_MISMATCH`, **no bundle is written**, and
the divergence is published as evidence.

**Boundary:** the bundle carries code, frozen configs, and the S7 fallback
package. The only data file is the development-only `scientific.parquet`
(train + validation). `test`, `stress`, and `monitor` remain sealed and
have no unlock path in this code. This package is frozen for confirmation,
not for deployment.

All computational logic lives in the shipped package
(`consumer_complaint_intelligence.kaggle_execution`); cells only
orchestrate and print aggregate evidence.


In [ ]:
%pip install --quiet scikit-learn==1.9.0 imbalanced-learn==0.14.2


In [ ]:
import sys
import zipfile
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/temp/project")
OUTPUT_ROOT = Path("/kaggle/working")


def _input_listing(limit=200):
    return [str(path) for path in sorted(INPUT_ROOT.rglob("*"))[:limit]]


manifests = [
    path
    for path in sorted(INPUT_ROOT.rglob("kaggle_bundle_manifest.json"))
    if (path.parent / "src").is_dir()
]
if manifests:
    bundle_root = manifests[0].parent
else:
    zips = sorted(INPUT_ROOT.rglob("cci-v2-bundle.zip"))
    if not zips:
        raise FileNotFoundError(
            f"No bundle manifest or zip under {INPUT_ROOT}; "
            f"mounted: {_input_listing()}"
        )
    bundle_root = Path("/kaggle/temp/bundle_extracted")
    bundle_root.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zips[0]) as archive:
        archive.extractall(bundle_root)

caches = sorted(INPUT_ROOT.rglob("scientific.parquet"))
if not caches:
    raise FileNotFoundError(
        f"No scientific.parquet under {INPUT_ROOT}; "
        f"mounted: {_input_listing()}"
    )
CACHE_FILE = caches[0]
print("bundle_root:", bundle_root)
print("cache_file:", CACHE_FILE)

sys.path.insert(0, str(bundle_root / "src"))
from consumer_complaint_intelligence import kaggle_execution as kx

print(kx.assert_pinned_environment()["installed"])


In [ ]:
staging = kx.stage_project(bundle_root, CACHE_FILE, WORK_ROOT)
print(staging)
print(kx.preflight_package(WORK_ROOT))


In [ ]:
result = kx.run_full_package(WORK_ROOT)
gate = result["reproduction_gate"]
print(
    {
        "outcome": result["outcome"],
        "frozen": result["frozen"],
        "complete": result["complete"],
        "runtime_seconds": result["runtime_seconds"],
        "gate_passed": gate["passed"],
        "failed_checks": gate["failed_checks"],
        "bundle_persisted": result["bundle"]["persisted"],
    }
)


In [ ]:
print(
    {
        "threshold": result["calibration"]["threshold"],
        "outer_effective_overrides": result["outer"]["effective_overrides"],
        "outer_critical_f1": result["outer"]["metrics"]["critical_f1"],
        "outer_critical_precision": (
            result["outer"]["metrics"]["critical_precision"]
        ),
        "outer_macro_f1": result["outer"]["metrics"]["macro_f1"],
        "hard_negative": result["hard_negative"],
    }
)
if gate["failed_checks"]:
    print("DIVERGENCES:", gate["divergences"])


In [ ]:
print(kx.collect_outputs_package(WORK_ROOT, OUTPUT_ROOT))


## Retrieval

The staged tree lives under `/kaggle/temp` and is discarded with the
session. Persisted as notebook output:

- `v2_package.json` -> local `temp/v2/`
- `v2_results.json` -> local `config/`
- `consumer_complaint_detector_v2.joblib` -> local `artifacts/v2/`
  (present only when the reproduction gate passed)

Download them (`kaggle kernels output`) and revalidate locally with
`v2_package.validate_v2_manifest` and the `tests/test_v2_*` suite before
the package is treated as frozen.
